# [Chapter 6 — SIR_I Analysis, §6.3] $R_0$ via the Next-Generation Matrix

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 6 — SIR_I Analysis
**Considerations developed:** 6 (R_0)
**Estimated runtime:** ≤ 10 seconds

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Computes $R_0$ as the spectral radius of the next-generation matrix $K = F V^{-1}$ for the SIR_I system. For this 1-class system the NGM reduces to a scalar equal to the heuristic prediction; the notebook demonstrates the method on a 2-class generalization to show how it scales.

## What you should already know
Chapter 6 heuristic-R_0 notebook; basic linear algebra.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
from shared.parameters import baseline_chapter_05
from shared.seeds import set_seed_chapter_06
from shared.verification import assert_R0_equivalence

set_seed_chapter_06()


## The next-generation matrix

For a compartmental epidemic model linearized at the disease-free equilibrium, decompose the infection-system Jacobian as $F - V$ where:
- $F$ is the matrix of *new infection* rates
- $V$ is the matrix of *transition* rates (recovery, mortality, progression)

The next-generation matrix is $K = F V^{-1}$, and:

$$R_0 = \rho(K) \quad \text{(spectral radius of } K\text{)}$$

For the basic SIR_I (one infected class, one transition), $K$ is a $1 \times 1$ matrix, and:
$$F = c_I \beta, \qquad V = 1/\tau_R, \qquad K = c_I \beta \tau_R = R_0$$

This gives the same answer as the heuristic, as it must.

In [ ]:
# Verify NGM gives same R_0 as heuristic for SIR_I
params = baseline_chapter_05()
c_I = params['c_I']
beta = params['beta']
tau_R = params['tau_R']

# Heuristic
R0_heuristic = c_I * beta * tau_R

# NGM (1x1 case for SIR_I)
F = np.array([[c_I * beta]])
V = np.array([[1.0 / tau_R]])
K = F @ np.linalg.inv(V)
R0_ngm = np.max(np.abs(np.linalg.eigvals(K)))

# Threshold (same expression for SIR_I)
R0_threshold = c_I * beta * tau_R

print(f"R_0 (heuristic):  {R0_heuristic:.6f}")
print(f"R_0 (NGM):        {R0_ngm:.6f}")
print(f"R_0 (threshold):  {R0_threshold:.6f}")

assert_R0_equivalence(R0_heuristic, R0_ngm, R0_threshold, rel_tol=1e-12)
print('\n✅ All three derivations agree to machine precision.')


## NGM in a 2-class generalization

For pedagogical purposes, here's the NGM for a two-group SIR (susceptible classes 1 and 2 with cross-mixing). This shows how the method generalizes — and why $R_0$ becomes the spectral radius of an actual matrix in non-trivial cases.

In [ ]:
# Two-group SIR with mixing matrix
# Group sizes
N1, N2 = 0.6, 0.4  # 60-40 split

# Within-group and between-group contact rates
c_I_within = 8.0
c_I_between = 2.0
tau_R = 10.0

# Mixing matrix (rows = from-class, cols = to-class)
beta_matrix = np.array([
    [c_I_within * beta * N1, c_I_between * beta * N1],
    [c_I_between * beta * N2, c_I_within * beta * N2]
])

V = np.eye(2) / tau_R

K = beta_matrix @ np.linalg.inv(V)
eigenvalues = np.linalg.eigvals(K)
R0_two_group = np.max(np.abs(eigenvalues))

print(f"Two-group K matrix:")
print(K)
print(f"\nEigenvalues: {eigenvalues}")
print(f"R_0 (spectral radius): {R0_two_group:.4f}")
print()
print(f"For comparison, well-mixed equivalent (averaged contact rate):")
c_I_avg = c_I_within * (N1**2 + N2**2) + c_I_between * 2 * N1 * N2
R0_well_mixed = c_I_avg * beta * tau_R
print(f"R_0 (well-mixed): {R0_well_mixed:.4f}")
print()
print(f"The two-group R_0 differs from well-mixed when contact patterns are heterogeneous.")
print(f"This is the foundation of Chapter 9 (two-group models).")
